### Tools
LLM Models can request to call tools that perform tasks such as fetching data from database, browsing the web,
or executing random task. Tools are pairings of:

1. A schema, including name of the tool, a description and argument definitions (JSON schema)
2. A function or coroutine to execute.

**We will use the LangChain's tool library and use the @tool decorator to create our tool.**

In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

True

In [9]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]


In [17]:

os.environ["GROQ_API_KEY"] = os.getenv("API_KEY_GROQ")
model = init_chat_model("groq:qwen/qwen3-32b")
if model and len(model.profile.keys()) != 0:
    print(f"Model loaded: {model}")
else:
    print(f"Unable to load the model. Please check the model name.")

Model loaded: profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x11068c6d0> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11068c290> model_name='qwen/qwen3-32b' model_kwargs={} groq_api_key=SecretStr('**********')


In [18]:
response = model.invoke("How strong are diamonds?")
response

AIMessage(content="<think>\nOkay, so I need to figure out how strong diamonds are. Let me start by recalling what I know about diamonds. I remember they're considered the hardest natural substance, right? But wait, there's a difference between hardness and strength. Maybe I should clarify those terms first. Hardness in minerals is usually measured on the Mohs scale, where 10 is the hardest. Diamonds are a 10 on that scale. Strength, on the other hand, might refer to other properties like tensile strength or toughness. \n\nSo, if diamonds are the hardest, does that mean they can't be scratched by anything else? But I've heard that diamonds can still be shattered if struck hard enough. So hardness and strength aren't the same. Hardness is resistance to scratching, while strength might be about how much force they can withstand before breaking. \n\nI think diamonds have high hardness but lower toughness. Toughness is the ability to resist breaking when a crack forms. Maybe because of thei

In [23]:
from langchain.tools import tool

# Use the tool decorator to make a function a tool.
# Make sure to provide docstring, because LLM will use this docstring to understand what
# this tool does.
# Here we can call an external API or retrieve information from database. 
@tool
def get_weather(location: str) -> str:
    """
        Get weather at the location provided by the user.        
    """
    return f"It's sunny in {location}"

# Bind the tool with model
model = model.bind_tools([get_weather])
model

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11068c6d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11068c290>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get weather at the location provided by the user.', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [24]:
response = model.invoke("What's weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'yj5fz6yne', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 156, 'total_tokens': 242, 'completion_time': 0.145451799, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.007843657, 'prompt_tokens_details': None, 'queue_time': 0.17476266, 'total_time': 0.153295456}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ca585-79ca-7f32-9a

In [ ]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Tool Args: {tool_call['args']}")

Tool: get_weather
Tool: {'location': 'Boston'}


### Tool Execution Loops
So instead of doing invoke on model, we are going to execute invoke on the tool itself.<br/>

**Step1: Model generates tool calls** -> Here we will define a *messages with user role and content* to call `invoke` on the model itself. Once we get the response back, we will append it the messages array.<br/>
**Step2: Execute tools and collect results**  -> Loop through all the tool calls and call `invoke` method on the tool itself to generate a response. We will append these individual responeses to the messages array.</br>
**Step3: Pass results of the tools back to the model for final response**



In [26]:
messages = [{"role": "user", "content": "What's the weather in NewYork?"}]
model_response = model.invoke(messages)
messages.append(model_response)
print(messages)

[{'role': 'user', 'content': "What's the weather in NewYork?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. I need to use the get_weather function. The function requires the location parameter. The user provided "NewYork" as the location. I should check if "NewYork" is a valid location. Since it\'s a well-known city, it\'s likely valid. I\'ll format the function call with the location parameter set to "New York", making sure to spell it correctly. Then, I\'ll return the tool_call JSON inside the XML tags.\n', 'tool_calls': [{'id': 'a60qfha8g', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 157, 'total_tokens': 285, 'completion_time': 0.229540189, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.007613328, 'prompt_tokens_details': None, 'queue_time': 0.007261198,

In [27]:
# Step 2:
for tool_call in model_response.tool_calls:
    #Execute tool with the generate arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

messages

[{'role': 'user', 'content': "What's the weather in NewYork?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. I need to use the get_weather function. The function requires the location parameter. The user provided "NewYork" as the location. I should check if "NewYork" is a valid location. Since it\'s a well-known city, it\'s likely valid. I\'ll format the function call with the location parameter set to "New York", making sure to spell it correctly. Then, I\'ll return the tool_call JSON inside the XML tags.\n', 'tool_calls': [{'id': 'a60qfha8g', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 157, 'total_tokens': 285, 'completion_time': 0.229540189, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.007613328, 'prompt_tokens_details': None, 'queue_time': 0.007261198

In [28]:
# Step 3: pass results back to model for final reponse
final_response = model.invoke(messages)
print(final_response)

content='The weather in New York is currently sunny. No rain is expected, and temperatures are warm. Let me know if you need more details! ☀️' additional_kwargs={'reasoning_content': 'Okay, the user asked for the weather in New York. I called the get_weather function with the location parameter set to "New York". The response came back saying it\'s sunny there. Now I need to relay that information back to the user in a friendly and clear way.\n\nFirst, I should confirm the location to make sure there\'s no confusion. Then, state the weather condition simply. Maybe add a sentence about what that means, like no rain and warm temperatures. Keep it concise but helpful. Avoid any technical jargon. Just a straightforward update on the weather as requested.\n'} response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 197, 'total_tokens': 350, 'completion_time': 0.310931175, 'completion_tokens_details': {'reasoning_tokens': 117}, 'prompt_time': 0.008223361, 'prompt_tokens_